# 03 · Experiments

The baseline scored **0.996** on validation (steps 29–35) but only **0.947** on the public leaderboard.
That validation was too easy (it only predicted 1–7 steps ahead), so it can't tell good ideas from bad ones.

This notebook:
1. Sets up a **harder, more realistic validation** that predicts ~10 steps ahead (closer to the 14-step test period).
2. Runs each improvement idea **one at a time** through the same `evaluate()` function and logs the result.


| Exp | Idea | Why it might help |
|---|---|---|
| A | Baseline LightGBM (raw 165 features) | Reference score |
| B | + graph structure (degrees, # usable neighbours) | Fraud transactions may have distinctive connectivity |
| C | + neighbours' feature averages | Fraud clusters with fraud (homophily check: 56–59% vs 3%) |
| D | + neighbours' **predicted** fraud probability (two-stage) | Uses homophily without needing neighbour labels |
| E | LightGBM + Random Forest average | RF is often more robust to drift over time |
| F | Adversarial validation | Finds features whose distribution shifts between train and test |
| G | Feature selection | Finds features which contribute more |

In [2]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"
SUB_DIR = "../submissions"
os.makedirs(SUB_DIR, exist_ok=True)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
train["label"] = pd.to_numeric(train["label"], errors="coerce")   # "none" -> NaN

feat_cols = [f"feat_{i}" for i in range(1, 166)]

## 1. Graph preparation



To keep train and test features consistent, every neighbour-based feature uses only
**"known" nodes = labelled training rows + test rows**. A training node's unlabelled neighbours are ignored,
exactly as they are invisible for test nodes.

Degrees are the exception: the edge list itself is complete for both periods, so degrees use all edges.

In [3]:
lab = train[train["label"].notna()].copy()
lab["label"] = lab["label"].astype(int)

# One table of all "known" nodes, indexed by txId
nodes = pd.concat([lab, test.drop(columns="index")], ignore_index=True).set_index("txId").copy()
nodes["is_test"] = nodes.index.isin(test["txId"])

# --- Structural features (full edge list) ---
out_deg = edges.groupby("txId1").size()      # how many later transactions spend this one's outputs
in_deg  = edges.groupby("txId2").size()      # how many earlier transactions feed into this one
nodes["in_deg"]  = in_deg.reindex(nodes.index).fillna(0)
nodes["out_deg"] = out_deg.reindex(nodes.index).fillna(0)
nodes["deg"]     = nodes["in_deg"] + nodes["out_deg"]

# --- Neighbour pairs restricted to known nodes, both directions ---
e = edges[edges["txId1"].isin(nodes.index) & edges["txId2"].isin(nodes.index)]
nbr = pd.concat([
    pd.DataFrame({"node": e["txId2"].values, "nbr": e["txId1"].values, "dir": "in"}),   # nbr pays into node
    pd.DataFrame({"node": e["txId1"].values, "nbr": e["txId2"].values, "dir": "out"}),  # node pays into nbr
], ignore_index=True)

n_known = nbr.groupby("node").size()
nodes["n_known_nbrs"]    = n_known.reindex(nodes.index).fillna(0)
nodes["frac_known_nbrs"] = nodes["n_known_nbrs"] / nodes["deg"].clip(lower=1)
STRUCT_COLS = ["in_deg", "out_deg", "deg", "n_known_nbrs", "frac_known_nbrs"]

# Sanity check: is coverage similar in train and test? (If not, these features mean different things.)
print(nodes.groupby("is_test")[["deg", "n_known_nbrs", "frac_known_nbrs"]].mean().round(3))
print("Share with ≥1 known neighbour:",
      nodes.groupby("is_test")["n_known_nbrs"].apply(lambda s: (s > 0).mean()).round(3).to_dict())

           deg  n_known_nbrs  frac_known_nbrs
is_test                                      
False    2.866          1.53            0.631
True     3.242          1.66            0.609
Share with ≥1 known neighbour: {False: 0.782, True: 0.746}


## 2. Harder validation

Expanding window, each fold predicting ~10 steps ahead (the real test is 14 steps ahead).
The folds overlap, which is fine: we only use the *mean* to compare experiments.

In [4]:
L = nodes[~nodes["is_test"]].copy()   # labelled training rows (with graph columns)
T = nodes[nodes["is_test"]].copy()    # test rows

FOLDS = [(15, range(16, 26)), (20, range(21, 31)), (25, range(26, 36))]   # (last train step, val steps)
LOG = []                                                                  # experiment log

def lgbm():
    return lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                              random_state=SEED, verbose=-1)

def raw_features(fit, apply):
    return fit[feat_cols], apply[feat_cols]

def evaluate(name, make_model=lgbm, build=raw_features):
    """For each fold: build features, train on past steps, score the future block.
    `build(fit, apply)` returns (X_fit, X_apply) so fold-dependent features can be created
    using only the training part of that fold."""
    aucs = []
    for last_train, val_steps in FOLDS:
        fit = L[L["time_step"] <= last_train]
        va  = L[L["time_step"].isin(val_steps)]
        X_fit, X_va = build(fit, va)
        model = make_model().fit(X_fit, fit["label"])
        aucs.append(roc_auc_score(va["label"], model.predict_proba(X_va)[:, 1]))
    LOG.append({"experiment": name, **{f"fold{i+1}": round(a, 4) for i, a in enumerate(aucs)},
                "mean_AUC": round(np.mean(aucs), 4)})
    print(f"{name:45s} folds={np.round(aucs, 4)}  mean={np.mean(aucs):.4f}")

evaluate("A  baseline LightGBM (raw)")

A  baseline LightGBM (raw)                    folds=[0.9594 0.9092 0.997 ]  mean=0.9552


**Check before continuing:** fold AUCs should now be noticeably below 0.99 and differ from each other.
If they're still ~0.99, this validation is still too easy; make the folds harder
(e.g. `(10, range(11, 25))`).

## 3. Experiment B: graph structure

In [5]:
def with_struct(fit, apply):
    cols = feat_cols + STRUCT_COLS
    return fit[cols], apply[cols]

evaluate("B  + structure (degrees, known nbrs)", build=with_struct)

B  + structure (degrees, known nbrs)          folds=[0.9554 0.8987 0.9971]  mean=0.9504


## 4. Experiment C: neighbours' feature averages

Averaging all 165 features over neighbours would add 165+ columns, so we aggregate only the
20 most important raw features. Importance is taken from a model trained on steps ≤ 15
(the first fold's training data) so the choice never sees the later validation labels.
Aggregates use only feature values (never labels), so they can be precomputed for all nodes.

In [6]:
imp_model = lgbm().fit(L.loc[L["time_step"] <= 15, feat_cols], L.loc[L["time_step"] <= 15, "label"])
TOP = pd.Series(imp_model.feature_importances_, index=feat_cols).nlargest(20).index.tolist()

nb_vals = nodes.loc[nbr["nbr"], TOP].set_index(nbr["node"].values)
agg = nb_vals.groupby(level=0).agg(["mean", "max"])
agg.columns = [f"nbr_{stat}_{c}" for c, stat in agg.columns]
NBR_AGG_COLS = list(agg.columns)
nodes = nodes.drop(columns=NBR_AGG_COLS, errors="ignore").join(agg)
nodes[NBR_AGG_COLS] = nodes[NBR_AGG_COLS].fillna(-999)   # -999 = "no known neighbour"
L, T = nodes[~nodes["is_test"]].copy(), nodes[nodes["is_test"]].copy()

def with_nbr_agg(fit, apply):
    cols = feat_cols + STRUCT_COLS + NBR_AGG_COLS
    return fit[cols], apply[cols]

evaluate("C  + neighbour feature mean/max (top 20)", build=with_nbr_agg)

C  + neighbour feature mean/max (top 20)      folds=[0.9551 0.896  0.9971]  mean=0.9494


## 5. Experiment D: neighbours' predicted fraud probability (two-stage)

Stage 1 predicts a fraud probability for every known node; stage 2 adds
"max / mean predicted probability among my neighbours" as features.

**Leakage control:** a training node's own label must not leak into its neighbours' features.
Neighbours always share a time step, so stage-1 predictions for training rows are made
**out-of-fold, grouped by time step**: each step is predicted by a model that never saw that step.
Rows being scored (validation or test) are predicted by a stage-1 model trained on all training rows.

In [6]:
def nbr_pred_features(fit, apply):
    """Neighbour-prediction features for every node in fit ∪ apply (index = txId)."""
    p_fit = pd.Series(np.nan, index=fit.index)
    for tr_idx, oof_idx in GroupKFold(n_splits=5).split(fit, groups=fit["time_step"]):
        m = lgbm().fit(fit.iloc[tr_idx][feat_cols], fit.iloc[tr_idx]["label"])
        p_fit.iloc[oof_idx] = m.predict_proba(fit.iloc[oof_idx][feat_cols])[:, 1]
    m = lgbm().fit(fit[feat_cols], fit["label"])
    p_apply = pd.Series(m.predict_proba(apply[feat_cols])[:, 1], index=apply.index)
    p = pd.concat([p_fit, p_apply])

    nb = nbr[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    out = nb.groupby("node")["p"].agg(["max", "mean"]).add_prefix("nbr_p_")
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()    # where money came from
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()   # where money went
    return out.reindex(p.index).fillna(-1)                                      # -1 = no known neighbour

def with_nbr_pred(fit, apply, base_cols=None):
    base_cols = base_cols or feat_cols + STRUCT_COLS
    f = nbr_pred_features(fit, apply)
    return (fit[base_cols].join(f.loc[fit.index]),
            apply[base_cols].join(f.loc[apply.index]))

evaluate("D  + neighbour predicted probability", build=with_nbr_pred)

D  + neighbour predicted probability          folds=[0.9552 0.8957 0.9978]  mean=0.9496


## 6. Experiment E: LightGBM + Random Forest average

In [7]:
class AvgEnsemble:
    """Averages predicted probabilities of several models."""
    def __init__(self, makers): self.makers = makers
    def fit(self, X, y):
        self.models = [mk().fit(X, y) for mk in self.makers]
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

def rf():
    return RandomForestClassifier(n_estimators=500, max_features=50, n_jobs=-1, random_state=SEED)

evaluate("E1 Random Forest alone (raw)", make_model=rf)
evaluate("E2 LightGBM + RF average (raw)", make_model=lambda: AvgEnsemble([lgbm, rf]))

E1 Random Forest alone (raw)                  folds=[0.9347 0.7786 0.988 ]  mean=0.9004
E2 LightGBM + RF average (raw)                folds=[0.9416 0.7823 0.9927]  mean=0.9055


In [15]:
def seed_avg(params=None, seeds=(0, 1, 2)):
    """LightGBM averaged over several random seeds (reduces run-to-run noise)."""
    params = params or {}
    return AvgEnsemble([lambda s=s: lgbm().set_params(random_state=s, **params) for s in seeds])

evaluate("A  baseline, 3-seed avg", make_model=seed_avg)
evaluate("D  + neighbour preds, 3-seed avg", make_model=seed_avg, build=with_nbr_pred)

grid = [
    dict(num_leaves=15, min_child_samples=50),
    dict(num_leaves=7,  min_child_samples=100),
    dict(colsample_bytree=0.3),
    dict(num_leaves=15, colsample_bytree=0.3, reg_lambda=5),
    dict(n_estimators=1000, learning_rate=0.015),
    dict(scale_pos_weight=5),
]
for p in grid:
    evaluate(f"H  {p}, 3-seed avg", make_model=lambda p=p: seed_avg(p))

A  baseline, 3-seed avg                       folds=[0.9598 0.9054 0.9969]  mean=0.9540
D  + neighbour preds, 3-seed avg              folds=[0.9589 0.8956 0.9979]  mean=0.9508
H  {'num_leaves': 15, 'min_child_samples': 50}, 3-seed avg folds=[0.9578 0.9083 0.9965]  mean=0.9542
H  {'num_leaves': 7, 'min_child_samples': 100}, 3-seed avg folds=[0.9562 0.9078 0.9953]  mean=0.9531
H  {'colsample_bytree': 0.3}, 3-seed avg      folds=[0.9564 0.9061 0.9971]  mean=0.9532
H  {'num_leaves': 15, 'colsample_bytree': 0.3, 'reg_lambda': 5}, 3-seed avg folds=[0.9439 0.8891 0.996 ]  mean=0.9430
H  {'n_estimators': 1000, 'learning_rate': 0.015}, 3-seed avg folds=[0.9605 0.9045 0.9969]  mean=0.9540
H  {'scale_pos_weight': 5}, 3-seed avg        folds=[0.9527 0.9013 0.9964]  mean=0.9501


In [16]:
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import QuantileTransformer
from sklearn.pipeline import make_pipeline

def mlp():
    """Two-layer neural network; inputs reshaped to a normal distribution first."""
    return make_pipeline(
        QuantileTransformer(output_distribution="normal", random_state=SEED),
        MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, early_stopping=True,
                      max_iter=200, random_state=SEED),
    )

class SVMScorer:
    """RBF SVM. SVMs output a score rather than a probability; AUC only needs the ranking,
    so the score is squashed to 0–1 and returned in predict_proba format."""
    def fit(self, X, y):
        self.m = make_pipeline(
            QuantileTransformer(output_distribution="normal", random_state=SEED),
            SVC(kernel="rbf", C=1.0, class_weight="balanced"),
        ).fit(X, y)
        return self
    def predict_proba(self, X):
        s = 1 / (1 + np.exp(-self.m.decision_function(X)))
        return np.column_stack([1 - s, s])

evaluate("I1 MLP (128-64)", make_model=mlp)
evaluate("I2 SVM (RBF)", make_model=SVMScorer)
evaluate("I3 LightGBM 3-seed + MLP average", make_model=lambda: AvgEnsemble([seed_avg, mlp]))

I1 MLP (128-64)                               folds=[0.9698 0.9455 0.9717]  mean=0.9623
I2 SVM (RBF)                                  folds=[0.9728 0.9238 0.9855]  mean=0.9607
I3 LightGBM 3-seed + MLP average              folds=[0.971  0.9402 0.9919]  mean=0.9677


In [17]:
from scipy.linalg import eigh
from sklearn.preprocessing import QuantileTransformer

def fraud_pca_features(fit, apply, k=10, reg=1e-3):
    """Directions where fraud varies much less than licit (smallest λ in S_fraud v = λ S_licit v).
    Fitted on the fold's training rows only, so validation labels are never used."""
    qt = QuantileTransformer(output_distribution="normal", random_state=SEED).fit(fit[feat_cols])
    Xf, Xa = qt.transform(fit[feat_cols]), qt.transform(apply[feat_cols])
    y = fit["label"].values
    S_fraud = np.cov(Xf[y == 1], rowvar=False)
    S_licit = np.cov(Xf[y == 0], rowvar=False)
    I = reg * np.eye(Xf.shape[1])                      # small ridge so the maths is stable
    lam, V = eigh(S_fraud + I, S_licit + I)            # eigenvalues come out smallest first
    W, mu = V[:, :k], Xf[y == 1].mean(axis=0)          # k "fraud-tight" directions, fraud centre

    def feats(X):
        z = (X - mu) @ W                               # offset from typical fraud along each direction
        return np.column_stack([z, (z ** 2).sum(axis=1)])
    cols = [f"fraud_pc_{i}" for i in range(k)] + ["fraud_pc_dist"]
    return (pd.DataFrame(feats(Xf), index=fit.index, columns=cols),
            pd.DataFrame(feats(Xa), index=apply.index, columns=cols))

def fraud_dist_only(fit, apply):
    pf, pa = fraud_pca_features(fit, apply)
    return pf[["fraud_pc_dist"]], pa[["fraud_pc_dist"]]

def with_fraud_pca(fit, apply, k=10):
    pf, pa = fraud_pca_features(fit, apply, k)
    return fit[feat_cols].join(pf), apply[feat_cols].join(pa)

evaluate("J0 fraud-PCA distance ALONE", build=fraud_dist_only)          # does it carry signal?
evaluate("J1 raw + fraud-PCA (k=10), 3-seed avg", make_model=seed_avg, build=with_fraud_pca)
evaluate("J2 raw + fraud-PCA (k=5), 3-seed avg",  make_model=seed_avg,
         build=lambda f, a: with_fraud_pca(f, a, k=5))

J0 fraud-PCA distance ALONE                   folds=[0.865  0.8396 0.9495]  mean=0.8847
J1 raw + fraud-PCA (k=10), 3-seed avg         folds=[0.9471 0.8532 0.9965]  mean=0.9323
J2 raw + fraud-PCA (k=5), 3-seed avg          folds=[0.957  0.8851 0.9964]  mean=0.9461


## 7. Experiment F: adversarial validation

Train a model to distinguish training rows from test rows. AUC ≈ 0.5 means the two periods look alike;
a high AUC means drift. The most useful features for telling them apart are drift suspects.
Dropping them is then tested with `evaluate()`; only keep the change if validation improves.

In [8]:
adv = pd.concat([L[feat_cols].assign(is_test=0), T[feat_cols].assign(is_test=1)])
adv_p = np.zeros(len(adv))
for tr_idx, va_idx in GroupKFold(n_splits=5).split(adv, groups=np.r_[L["time_step"], T["time_step"]]):
    m = lgbm().fit(adv.iloc[tr_idx][feat_cols], adv.iloc[tr_idx]["is_test"])
    adv_p[va_idx] = m.predict_proba(adv.iloc[va_idx][feat_cols])[:, 1]
print(f"Adversarial AUC (train vs test): {roc_auc_score(adv['is_test'], adv_p):.3f}")

m = lgbm().fit(adv[feat_cols], adv["is_test"])
drift = pd.Series(m.feature_importances_, index=feat_cols).sort_values(ascending=False)
print("Top drift suspects:", drift.head(10).index.tolist())

for k in [5, 10, 20]:
    keep = [c for c in feat_cols if c not in drift.head(k).index]
    evaluate(f"F  drop top-{k} drift features",
             build=lambda fit, apply, keep=keep: (fit[keep], apply[keep]))

Adversarial AUC (train vs test): 0.999
Top drift suspects: ['feat_139', 'feat_136', 'feat_100', 'feat_2', 'feat_138', 'feat_101', 'feat_137', 'feat_103', 'feat_90', 'feat_154']
F  drop top-5 drift features                  folds=[0.9587 0.9054 0.9976]  mean=0.9539
F  drop top-10 drift features                 folds=[0.9545 0.9007 0.9973]  mean=0.9508
F  drop top-20 drift features                 folds=[0.9553 0.8743 0.9973]  mean=0.9423


## 8. Experiment G: Feature Selection

In [12]:
import shap

# Rank features using a model trained ONLY on steps ≤ 15. Every validation fold
# starts at step 16 or later, so this ranking never sees validation labels.
fit = L[L["time_step"] <= 15]
m = lgbm().fit(fit[feat_cols], fit["label"])
sv = shap.TreeExplainer(m).shap_values(fit[feat_cols])
sv = sv[1] if isinstance(sv, list) else sv          # some shap versions return [class0, class1]
shap_rank = pd.Series(np.abs(sv).mean(axis=0), index=feat_cols).sort_values(ascending=False)
print(shap_rank.head(15))

for k in [30, 60, 100]:
    keep = shap_rank.head(k).index.tolist()
    evaluate(f"G  top-{k} SHAP features", build=lambda fit, apply, keep=keep: (fit[keep], apply[keep]))

ModuleNotFoundError: No module named 'shap'

## 8. Results

In [11]:
log = pd.DataFrame(LOG).sort_values("mean_AUC", ascending=False).reset_index(drop=True)
log

,experiment,fold1,fold2,fold3,mean_AUC
0,A baseline LightGBM (raw),0.9594,0.9092,0.9970,0.9552
1,F drop top-5 drift features,0.9587,0.9054,0.9976,0.9539
2,F drop top-10 drift features,0.9545,0.9007,0.9973,0.9508
3,"B + structure (degrees, known nbrs)",0.9554,0.8987,0.9971,0.9504
4,D + neighbour predicted probability,0.9552,0.8957,0.9978,0.9496
5,C + neighbour feature mean/max (top 20),0.9551,0.8960,0.9971,0.9494
6,F drop top-20 drift features,0.9553,0.8743,0.9973,0.9423
7,E2 LightGBM + RF average (raw),0.9416,0.7823,0.9927,0.9055
8,E1 Random Forest alone (raw),0.9347,0.7786,0.9880,0.9004


In [13]:
for s in range(5):
    evaluate(f"A  baseline seed {s}", make_model=lambda s=s: lgbm().set_params(random_state=s))

A  baseline seed 0                            folds=[0.9581 0.8906 0.9966]  mean=0.9485
A  baseline seed 1                            folds=[0.9616 0.9129 0.997 ]  mean=0.9572
A  baseline seed 2                            folds=[0.9579 0.9053 0.9968]  mean=0.9533
A  baseline seed 3                            folds=[0.9608 0.8904 0.9967]  mean=0.9493
A  baseline seed 4                            folds=[0.9616 0.8926 0.9966]  mean=0.9503


## 9. Submission for a chosen configuration

Set `MODEL` and `BUILD` to the best combination from the table, then run.
Features are rebuilt with **all** labelled rows as the fit set and the test rows as the apply set.

In [18]:
MODEL = lambda: AvgEnsemble([seed_avg, mlp])
BUILD, TAG = raw_features, "03_lgbm3seed_mlp_avg"

X_fit, X_test = BUILD(L, T)
final = MODEL().fit(X_fit, L["label"])
sub = pd.DataFrame({"index": test["index"].values,
                    "target": final.predict_proba(X_test.loc[test["txId"]])[:, 1]})

sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(sample) and set(sub["index"]) == set(sample["index"])
assert sub["target"].between(0, 1).all()

sub.to_csv(f"{SUB_DIR}/{TAG}.csv", index=False)
print(f"Saved {SUB_DIR}/{TAG}.csv")

Saved ../submissions/03_lgbm3seed_mlp_avg.csv


In [19]:
def mlp_seed(s):
    return lambda: mlp().set_params(mlpclassifier__random_state=s)

def mlp_avg(seeds=(0, 1, 2)):
    return AvgEnsemble([mlp_seed(s) for s in seeds])

class WeightedEnsemble(AvgEnsemble):
    """Weighted average of model probabilities."""
    def __init__(self, makers, weights):
        super().__init__(makers)
        self.w = np.array(weights) / np.sum(weights)
    def predict_proba(self, X):
        return np.tensordot(self.w, [m.predict_proba(X) for m in self.models], axes=1)

for s in range(3):
    evaluate(f"I1 MLP seed {s}", make_model=mlp_seed(s))
evaluate("I4 MLP 3-seed avg", make_model=mlp_avg)

for w in [0.3, 0.5, 0.7]:
    evaluate(f"I5 LGBM {1-w:.1f} / MLP {w:.1f} (3 seeds each)",
             make_model=lambda w=w: WeightedEnsemble([seed_avg, mlp_avg], [1 - w, w]))

evaluate("I6 LGBM + MLP + SVM (equal)", make_model=lambda: AvgEnsemble([seed_avg, mlp_avg, SVMScorer]))

I1 MLP seed 0                                 folds=[0.9738 0.926  0.9849]  mean=0.9616
I1 MLP seed 1                                 folds=[0.9752 0.936  0.9842]  mean=0.9651
I1 MLP seed 2                                 folds=[0.9669 0.9297 0.9837]  mean=0.9601
I4 MLP 3-seed avg                             folds=[0.9735 0.928  0.9853]  mean=0.9623
I5 LGBM 0.7 / MLP 0.3 (3 seeds each)          folds=[0.9749 0.9239 0.9945]  mean=0.9645
I5 LGBM 0.5 / MLP 0.5 (3 seeds each)          folds=[0.9747 0.9261 0.9936]  mean=0.9648
I5 LGBM 0.3 / MLP 0.7 (3 seeds each)          folds=[0.9744 0.9273 0.9921]  mean=0.9646
I6 LGBM + MLP + SVM (equal)                   folds=[0.9754 0.9261 0.993 ]  mean=0.9648


In [20]:
from scipy.stats import spearmanr

lg = seed_avg().fit(L[feat_cols], L["label"]).predict_proba(T[feat_cols])[:, 1]
ml = mlp_avg().fit(L[feat_cols], L["label"]).predict_proba(T[feat_cols])[:, 1]

print(f"Rank correlation LGBM vs MLP on test: {spearmanr(lg, ml).correlation:.3f}")
for name, p in [("LGBM", lg), ("MLP", ml)]:
    print(f"{name}: <0.01 = {(p < 0.01).mean():.1%} | >0.99 = {(p > 0.99).mean():.1%} | "
          f"flagged >0.5 = {(p > 0.5).mean():.1%}")

rows = []
for t, g in T[["time_step"]].assign(lg=lg, ml=ml).groupby("time_step"):
    rows.append({"step": t, "n": len(g), "lgbm_flag": (g.lg > 0.5).mean(),
                 "mlp_flag": (g.ml > 0.5).mean(), "rank_corr": spearmanr(g.lg, g.ml).correlation})
pd.DataFrame(rows).round(3)

Rank correlation LGBM vs MLP on test: 0.670
LGBM: <0.01 = 91.9% | >0.99 = 3.0% | flagged >0.5 = 4.0%
MLP: <0.01 = 90.4% | >0.99 = 0.8% | flagged >0.5 = 3.6%


,step,n,lgbm_flag,mlp_flag,rank_corr
0,36,1708,0.019,0.018,0.475
1,37,498,0.052,0.040,0.522
2,38,756,0.132,0.124,0.785
3,39,1183,0.060,0.057,0.649
4,40,1211,0.059,0.053,0.788
5,41,1132,0.095,0.089,0.660
6,42,2154,0.087,0.084,0.752
7,43,1370,0.000,0.000,0.709
8,44,1591,0.004,0.000,0.716
9,45,1221,0.001,0.001,0.783


In [21]:
from scipy.stats import rankdata

class RankAvgEnsemble(AvgEnsemble):
    """Averages each model's ranking (0–1) instead of raw probabilities,
    so no model dominates just because its probabilities are more extreme."""
    def predict_proba(self, X):
        r = np.mean([rankdata(m.predict_proba(X)[:, 1]) / len(X) for m in self.models], axis=0)
        return np.column_stack([1 - r, r])

evaluate("I7 LGBM + MLP rank average", make_model=lambda: RankAvgEnsemble([seed_avg, mlp_avg]))

I7 LGBM + MLP rank average                    folds=[0.9701 0.9212 0.9958]  mean=0.9624


In [22]:
class SelfTrain:
    """Transductive self-training: fit, pseudo-label the most confident rows of the data being
    scored, refit on train + pseudo-labels, and predict again. Uses features of the scored rows only."""
    def __init__(self, maker, top_pos=0.02, top_neg=0.5):
        self.maker, self.top_pos, self.top_neg = maker, top_pos, top_neg
    def fit(self, X, y):
        self.X, self.y = X, np.asarray(y)
        self.base = self.maker().fit(X, y)
        return self
    def predict_proba(self, X):
        order = np.argsort(self.base.predict_proba(X)[:, 1])
        n_pos = max(1, int(self.top_pos * len(X)))      # most confident 2% -> labelled fraud
        n_neg = int(self.top_neg * len(X))              # least suspicious 50% -> labelled licit
        pos, neg = order[len(order) - n_pos:], order[:n_neg]
        X2 = pd.concat([self.X, X.iloc[pos], X.iloc[neg]])
        y2 = np.r_[self.y, np.ones(len(pos)), np.zeros(len(neg))]
        return self.maker().fit(X2, y2).predict_proba(X)

evaluate("M1 self-training LGBM (3-seed)", make_model=lambda: SelfTrain(seed_avg))
evaluate("M2 self-training MLP (3-seed)",  make_model=lambda: SelfTrain(mlp_avg))
evaluate("M3 self-training LGBM+MLP rank avg",
         make_model=lambda: SelfTrain(lambda: RankAvgEnsemble([seed_avg, mlp_avg])))

M1 self-training LGBM (3-seed)                folds=[0.9597 0.8966 0.997 ]  mean=0.9511
M2 self-training MLP (3-seed)                 folds=[0.9499 0.8996 0.9871]  mean=0.9455
M3 self-training LGBM+MLP rank avg            folds=[0.9678 0.9142 0.9959]  mean=0.9593


In [23]:
from scipy.stats import rankdata

def within_step_rank(p, steps):
    """Rank each prediction among rows of the same time step (0–1)."""
    out = np.empty(len(p))
    for t in np.unique(steps):
        m = steps == t
        out[m] = rankdata(p[m]) / m.sum()
    return out

steps = T["time_step"].values
global_rank = (rankdata(lg) + rankdata(ml)) / (2 * len(lg))                          # LGBM+MLP, ranked across all test rows
step_rank   = (within_step_rank(lg, steps) + within_step_rank(ml, steps)) / 2       # ranked within each step

for alpha, tag in [(1.0, "06_rankavg_global"), (0.5, "07_rankavg_half_step")]:
    score = alpha * global_rank + (1 - alpha) * step_rank
    sub = pd.DataFrame({"index": test["index"].values,
                        "target": pd.Series(score, index=T.index).loc[test["txId"]].values})
    sub.to_csv(f"{SUB_DIR}/{tag}.csv", index=False)
    print(f"Saved {tag}.csv")

Saved 06_rankavg_global.csv
Saved 07_rankavg_half_step.csv


In [24]:
class RecentOnly:
    """Train only on the most recent `window` time steps of the training data."""
    def __init__(self, maker, window=15):
        self.maker, self.window = maker, window
    def fit(self, X, y):
        t = L.loc[X.index, "time_step"]
        keep = t > t.max() - self.window
        self.m = self.maker().fit(X[keep], np.asarray(y)[keep.values])
        return self
    def predict_proba(self, X):
        return self.m.predict_proba(X)

evaluate("P1 LGBM 3-seed, last 15 steps only", make_model=lambda: RecentOnly(seed_avg, 15))
evaluate("P2 LGBM 3-seed, last 10 steps only", make_model=lambda: RecentOnly(seed_avg, 10))

P1 LGBM 3-seed, last 15 steps only            folds=[0.9598 0.9023 0.9975]  mean=0.9532
P2 LGBM 3-seed, last 10 steps only            folds=[0.9534 0.8867 0.9976]  mean=0.9459


In [25]:
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

def xgb(seed=SEED):
    return XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8,
                         colsample_bytree=0.5, random_state=seed, n_jobs=-1, eval_metric="auc")

def cat(seed=SEED):
    return CatBoostClassifier(iterations=800, learning_rate=0.05, depth=6,
                              random_seed=seed, verbose=0)

evaluate("O1 XGBoost", make_model=xgb)
evaluate("O2 CatBoost", make_model=cat)
evaluate("O3 LGBM 3-seed + XGBoost", make_model=lambda: AvgEnsemble([seed_avg, xgb]))
evaluate("O4 LGBM 3-seed + CatBoost", make_model=lambda: AvgEnsemble([seed_avg, cat]))
evaluate("O5 LGBM + XGB + CatBoost", make_model=lambda: AvgEnsemble([seed_avg, xgb, cat]))

O1 XGBoost                                    folds=[0.9524 0.8827 0.9962]  mean=0.9438
O2 CatBoost                                   folds=[0.9484 0.835  0.9967]  mean=0.9267
O3 LGBM 3-seed + XGBoost                      folds=[0.9524 0.8836 0.9968]  mean=0.9443
O4 LGBM 3-seed + CatBoost                     folds=[0.9486 0.8412 0.997 ]  mean=0.9289
O5 LGBM + XGB + CatBoost                      folds=[0.9508 0.8674 0.997 ]  mean=0.9384


In [26]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import QuantileTransformer

U = train[train["label"].isna()].set_index("txId")          # ~110k unlabelled training transactions

# Graph neighbour pairs in both directions (used to look up labelled neighbours)
NB_ALL = pd.concat([edges.rename(columns={"txId1": "node", "txId2": "nbr"}),
                    edges.rename(columns={"txId2": "node", "txId1": "nbr"})], ignore_index=True)

class LGBMSeeds:
    """LightGBM averaged over seeds, with sample-weight support."""
    def __init__(self, seeds=(0, 1, 2)):
        self.seeds = seeds
    def fit(self, X, y, sample_weight=None):
        self.models = [lgbm().set_params(random_state=s).fit(X, y, sample_weight=sample_weight)
                       for s in self.seeds]
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

class PseudoLabelUnlabelled:
    """Adds confidently pseudo-labelled UNLABELLED training transactions, then retrains.
    A row is pseudo-labelled only when three signals agree: the model's prediction, the labels of its
    k most similar labelled transactions within ±window time steps, and its labelled graph neighbours.
    Only unlabelled rows up to the latest training step are used (no future information)."""
    def __init__(self, k=10, window=1, weight=0.5, pos_thr=0.9, neg_thr=0.01,
                 knn_pos=0.8, neg_cap=1.0, seeds=(0, 1, 2)):
        self.__dict__.update(locals()); del self.__dict__["self"]

    def fit(self, X, y):
        cols, y = X.columns, np.asarray(y)
        t_fit = L.loc[X.index, "time_step"].values
        pool = U[U["time_step"] <= t_fit.max()]

        # Signal 1: model prediction
        p_model = LGBMSeeds(self.seeds).fit(X, y).predict_proba(pool[cols])[:, 1]

        # Signal 2: k nearest labelled transactions within ±window steps (features reshaped to normal)
        qt = QuantileTransformer(output_distribution="normal", random_state=SEED).fit(X)
        Xq, Pq = qt.transform(X), qt.transform(pool[cols])
        t_pool = pool["time_step"].values
        p_knn = np.full(len(pool), np.nan)
        for t in np.unique(t_pool):
            ref = np.abs(t_fit - t) <= self.window
            m = t_pool == t
            if ref.sum() < self.k:
                continue
            if y[ref].sum() == 0:
                p_knn[m] = 0.0
                continue
            knn = KNeighborsClassifier(n_neighbors=self.k).fit(Xq[ref], y[ref])
            p_knn[m] = knn.predict_proba(Pq[m])[:, list(knn.classes_).index(1)]

        # Signal 3: share of illicit among labelled graph neighbours (NaN if none)
        nb = NB_ALL[NB_ALL["node"].isin(pool.index) & NB_ALL["nbr"].isin(X.index)].copy()
        nb["y"] = pd.Series(y, index=X.index).reindex(nb["nbr"]).values
        nbr_mean = nb.groupby("node")["y"].mean().reindex(pool.index).values

        # Agreement rules (a NaN neighbour share = "no graph evidence", which doesn't block)
        pos = (p_model >= self.pos_thr) & (p_knn >= self.knn_pos) & ~(nbr_mean < 0.5)
        neg = (p_model <= self.neg_thr) & (p_knn == 0) & ~(nbr_mean > 0)
        neg_idx = np.flatnonzero(neg)
        cap = int(self.neg_cap * len(y))                 # don't let pseudo-licit rows swamp real data
        if len(neg_idx) > cap:
            neg_idx = np.random.default_rng(SEED).choice(neg_idx, cap, replace=False)
        pos_idx = np.flatnonzero(pos)

        X2 = pd.concat([X, pool.iloc[pos_idx][cols], pool.iloc[neg_idx][cols]])
        y2 = np.r_[y, np.ones(len(pos_idx)), np.zeros(len(neg_idx))]
        w = np.r_[np.ones(len(y)), np.full(len(pos_idx) + len(neg_idx), self.weight)]
        print(f"   steps ≤{t_fit.max()}: real {len(y)} (illicit {int(y.sum())}) | "
              f"+{len(pos_idx)} pseudo-illicit, +{len(neg_idx)} pseudo-licit")
        self.model = LGBMSeeds(self.seeds).fit(X2, y2, sample_weight=w)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)

evaluate("Q0 LGBM 3-seed (reference, same class)", make_model=LGBMSeeds)
evaluate("Q1 pseudo-label unlabelled (weight 0.5)", make_model=PseudoLabelUnlabelled)
evaluate("Q2 pseudo-label unlabelled (weight 1.0)", make_model=lambda: PseudoLabelUnlabelled(weight=1.0))
evaluate("Q3 pseudo-label, stricter (model≥0.95, kNN 10/10)",
         make_model=lambda: PseudoLabelUnlabelled(pos_thr=0.95, knn_pos=1.0))

Q0 LGBM 3-seed (reference, same class)        folds=[0.9598 0.9054 0.9969]  mean=0.9540
   steps ≤15: real 15514 (illicit 1152) | +623 pseudo-illicit, +15514 pseudo-licit
   steps ≤20: real 18889 (illicit 1771) | +1219 pseudo-illicit, +18889 pseudo-licit
   steps ≤25: real 24200 (illicit 2337) | +1446 pseudo-illicit, +24200 pseudo-licit
Q1 pseudo-label unlabelled (weight 0.5)       folds=[0.9384 0.8203 0.9972]  mean=0.9186
   steps ≤15: real 15514 (illicit 1152) | +623 pseudo-illicit, +15514 pseudo-licit
   steps ≤20: real 18889 (illicit 1771) | +1219 pseudo-illicit, +18889 pseudo-licit
   steps ≤25: real 24200 (illicit 2337) | +1446 pseudo-illicit, +24200 pseudo-licit
Q2 pseudo-label unlabelled (weight 1.0)       folds=[0.9297 0.818  0.9973]  mean=0.9150
   steps ≤15: real 15514 (illicit 1152) | +397 pseudo-illicit, +15514 pseudo-licit
   steps ≤20: real 18889 (illicit 1771) | +948 pseudo-illicit, +18889 pseudo-licit
   steps ≤25: real 24200 (illicit 2337) | +1050 pseudo-illicit, +242

In [27]:
evaluate("Q4 pseudo-illicit only (no pseudo-licit)",
         make_model=lambda: PseudoLabelUnlabelled(neg_cap=0.0))
evaluate("Q5 pseudo-licit capped at 10% of real rows",
         make_model=lambda: PseudoLabelUnlabelled(neg_cap=0.1))

   steps ≤15: real 15514 (illicit 1152) | +623 pseudo-illicit, +0 pseudo-licit
   steps ≤20: real 18889 (illicit 1771) | +1219 pseudo-illicit, +0 pseudo-licit
   steps ≤25: real 24200 (illicit 2337) | +1446 pseudo-illicit, +0 pseudo-licit
Q4 pseudo-illicit only (no pseudo-licit)      folds=[0.9631 0.8932 0.9969]  mean=0.9511
   steps ≤15: real 15514 (illicit 1152) | +623 pseudo-illicit, +1551 pseudo-licit
   steps ≤20: real 18889 (illicit 1771) | +1219 pseudo-illicit, +1888 pseudo-licit
   steps ≤25: real 24200 (illicit 2337) | +1446 pseudo-illicit, +2420 pseudo-licit
Q5 pseudo-licit capped at 10% of real rows    folds=[0.9483 0.8761 0.9968]  mean=0.9404


In [28]:
from scipy.stats import spearmanr

sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
base = pd.read_csv(f"{SUB_DIR}/01_baseline_LightGBM.csv").set_index("index")["target"]

def make_submission(make_model, build, tag):
    """Train on ALL labelled rows (steps 1–35), predict the test set, save the CSV,
    and report how similar the ranking is to the 01 baseline submission."""
    X_fit, X_test = build(L, T)
    model = make_model().fit(X_fit, L["label"])
    p = model.predict_proba(X_test.loc[test["txId"]])[:, 1]
    sub = pd.DataFrame({"index": test["index"].values, "target": p})
    assert len(sub) == len(sample) and set(sub["index"]) == set(sample["index"])
    assert sub["target"].between(0, 1).all()
    sub.to_csv(f"{SUB_DIR}/{tag}.csv", index=False)
    corr = spearmanr(sub.set_index("index")["target"], base.loc[sub["index"]]).correlation
    print(f"{tag:28s} saved | rank correlation with 01 baseline: {corr:.4f}")

lgbm5 = lambda: seed_avg(seeds=(0, 1, 2, 3, 4))       # LightGBM averaged over 5 seeds

In [29]:
keep_20 = [c for c in feat_cols if c not in drift.head(20).index]
make_submission(lgbm5, lambda f, a: (f[keep_20], a[keep_20]), "11_lgbm5_drop20drift")

11_lgbm5_drop20drift         saved | rank correlation with 01 baseline: 0.9737


In [30]:
make_submission(lgbm5, with_nbr_pred, "10_lgbm5_nbrpred")

10_lgbm5_nbrpred             saved | rank correlation with 01 baseline: 0.9675


In [31]:
make_submission(rf, raw_features, "12_rf")

12_rf                        saved | rank correlation with 01 baseline: 0.7213


In [32]:
blk = pd.Series(np.where([int(c.split("_")[1]) <= 93 for c in drift.index], "feat_1–93", "feat_94–165"),
                index=drift.index)
print("Share of total drift importance:", (drift.groupby(blk).sum() / drift.sum()).round(3).to_dict())
print("Among top-30 drift features:     ", blk[drift.head(30).index].value_counts().to_dict())

Share of total drift importance: {'feat_1–93': 0.385, 'feat_94–165': 0.615}
Among top-30 drift features:      {'feat_94–165': 18, 'feat_1–93': 12}


In [33]:
first93 = [f"feat_{i}" for i in range(1, 94)]
evaluate("R1 LGBM 3-seed, columns 1–93 only", make_model=seed_avg,
         build=lambda f, a: (f[first93], a[first93]))

R1 LGBM 3-seed, columns 1–93 only             folds=[0.9532 0.8762 0.9952]  mean=0.9415


In [34]:
def stage1_preds(fit, apply):
    """Fraud probability for every node in fit ∪ apply. Training rows are predicted
    out-of-fold grouped by time step, so no node's own label leaks into its neighbours' scores."""
    p_fit = pd.Series(np.nan, index=fit.index)
    for tr_idx, oof_idx in GroupKFold(n_splits=5).split(fit, groups=fit["time_step"]):
        m = lgbm().fit(fit.iloc[tr_idx][feat_cols], fit.iloc[tr_idx]["label"])
        p_fit.iloc[oof_idx] = m.predict_proba(fit.iloc[oof_idx][feat_cols])[:, 1]
    m = lgbm().fit(fit[feat_cols], fit["label"])
    p_apply = pd.Series(m.predict_proba(apply[feat_cols])[:, 1], index=apply.index)
    return pd.concat([p_fit, p_apply])

def nbr_pred_features_v2(fit, apply):
    """1-hop and 2-hop neighbour-prediction features."""
    p = stage1_preds(fit, apply)
    nb = nbr[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    g = nb.groupby("node")["p"]
    out = pd.DataFrame({
        "nbr_p_max": g.max(), "nbr_p_mean": g.mean(), "nbr_p_sum": g.sum(),
        "nbr_n_high": nb.assign(h=nb["p"] > 0.5).groupby("node")["h"].sum(),   # # suspicious neighbours
    })
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()    # where money came from
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()   # where money went

    # 2-hop: neighbours of neighbours, excluding the node itself
    pairs = nb[["node", "nbr"]]
    two = pairs.merge(pairs.rename(columns={"node": "nbr", "nbr": "nbr2"}), on="nbr")
    two = two[two["nbr2"] != two["node"]].drop_duplicates(["node", "nbr2"])
    two["p"] = p.reindex(two["nbr2"]).values
    g2 = two.groupby("node")["p"]
    out["nbr2_p_max"], out["nbr2_p_mean"], out["nbr2_n"] = g2.max(), g2.mean(), g2.size()
    return out.reindex(p.index).fillna(-1)                                       # -1 = no such neighbour

def with_nbr_pred_v2(fit, apply):
    base = feat_cols + STRUCT_COLS
    f = nbr_pred_features_v2(fit, apply)
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

# Sanity check on validation: it shouldn't be clearly worse than D (0.951)
evaluate("D2 + 1-hop & 2-hop neighbour preds, 3-seed", make_model=seed_avg, build=with_nbr_pred_v2)

# Submission file
make_submission(lgbm5, with_nbr_pred_v2, "17_lgbm5_nbrpred_2hop")

D2 + 1-hop & 2-hop neighbour preds, 3-seed    folds=[0.9597 0.8904 0.9983]  mean=0.9494
17_lgbm5_nbrpred_2hop        saved | rank correlation with 01 baseline: 0.9604


In [35]:
p10 = pd.read_csv(f"{SUB_DIR}/10_lgbm5_nbrpred.csv").set_index("index")["target"]
p17 = pd.read_csv(f"{SUB_DIR}/17_lgbm5_nbrpred_2hop.csv").set_index("index")["target"]
print(f"Rank correlation 17 vs 10: {spearmanr(p17, p10.loc[p17.index]).correlation:.4f}")

Rank correlation 17 vs 10: 0.9920


In [36]:
def agg_nbr(p):
    """1-hop and 2-hop neighbour aggregates of a score p (Series indexed by txId)."""
    nb = nbr[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    g = nb.groupby("node")["p"]
    out = pd.DataFrame({"nbr_p_max": g.max(), "nbr_p_mean": g.mean(), "nbr_p_sum": g.sum(),
                        "nbr_n_high": nb.assign(h=nb["p"] > 0.5).groupby("node")["h"].sum()})
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()
    pairs = nb[["node", "nbr"]]
    two = pairs.merge(pairs.rename(columns={"node": "nbr", "nbr": "nbr2"}), on="nbr")
    two = two[two["nbr2"] != two["node"]].drop_duplicates(["node", "nbr2"])
    two["p"] = p.reindex(two["nbr2"]).values
    g2 = two.groupby("node")["p"]
    out["nbr2_p_max"], out["nbr2_p_mean"], out["nbr2_n"] = g2.max(), g2.mean(), g2.size()
    return out.reindex(p.index).fillna(-1)

def oof_then_apply(X_fit, y, groups, X_apply):
    """Out-of-fold predictions for training rows (grouped by time step), plus predictions for apply rows."""
    p_fit = pd.Series(np.nan, index=X_fit.index)
    for tr, te in GroupKFold(n_splits=5).split(X_fit, groups=groups):
        m = lgbm().fit(X_fit.iloc[tr], y.iloc[tr])
        p_fit.iloc[te] = m.predict_proba(X_fit.iloc[te])[:, 1]
    m = lgbm().fit(X_fit, y)
    return pd.concat([p_fit, pd.Series(m.predict_proba(X_apply)[:, 1], index=X_apply.index)])

def with_iterative(fit, apply, rounds=2):
    """Round 1 scores nodes from raw features; each extra round re-scores them using neighbours' scores."""
    base = feat_cols + STRUCT_COLS
    p = oof_then_apply(fit[feat_cols], fit["label"], fit["time_step"], apply[feat_cols])
    for _ in range(rounds - 1):
        f = agg_nbr(p)
        p = oof_then_apply(fit[base].join(f.loc[fit.index]), fit["label"], fit["time_step"],
                           apply[base].join(f.loc[apply.index]))
    f = agg_nbr(p)
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

def with_all_graph(fit, apply):
    """Raw + structure + neighbour feature averages (C) + 1-hop/2-hop neighbour predictions."""
    base = feat_cols + STRUCT_COLS + NBR_AGG_COLS
    f = nbr_pred_features_v2(fit, apply)
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

# Safety checks on validation (should not be clearly below ~0.949)
evaluate("D3 all graph features (C + 1/2-hop preds), 3-seed", make_model=seed_avg, build=with_all_graph)
evaluate("D4 iterative neighbour preds (2 rounds), 3-seed",   make_model=seed_avg, build=with_iterative)

# Submission files
make_submission(lgbm5, with_all_graph, "18_lgbm5_allgraph")
make_submission(lgbm5, with_iterative, "19_lgbm5_iter2")

D3 all graph features (C + 1/2-hop preds), 3-seed folds=[0.9595 0.8894 0.9982]  mean=0.9490
D4 iterative neighbour preds (2 rounds), 3-seed folds=[0.9615 0.889  0.9983]  mean=0.9496
18_lgbm5_allgraph            saved | rank correlation with 01 baseline: 0.9487
19_lgbm5_iter2               saved | rank correlation with 01 baseline: 0.9600


In [37]:
p17 = pd.read_csv(f"{SUB_DIR}/17_lgbm5_nbrpred_2hop.csv").set_index("index")["target"]
for tag in ["18_lgbm5_allgraph", "19_lgbm5_iter2"]:
    p = pd.read_csv(f"{SUB_DIR}/{tag}.csv").set_index("index")["target"]
    print(f"{tag:22s} rank correlation with 17: {spearmanr(p, p17.loc[p.index]).correlation:.4f}")

18_lgbm5_allgraph      rank correlation with 17: 0.9849
19_lgbm5_iter2         rank correlation with 17: 0.9953


In [38]:
import networkx as nx
import scipy.sparse as sp

# --- Full transaction graph from the COMPLETE edge list (includes unlabelled "stepping stone" nodes) ---
ALL_IDS = pd.Index(pd.unique(edges[["txId1", "txId2"]].values.ravel()))
POS = pd.Series(np.arange(len(ALL_IDS)), index=ALL_IDS)
_r, _c = POS[edges["txId1"]].values, POS[edges["txId2"]].values
_A = sp.coo_matrix((np.ones(len(_r)), (_r, _c)), shape=(len(ALL_IDS),) * 2).tocsr()
_U = ((_A + _A.T) > 0).astype(float)                       # undirected: fraud proximity works both ways
_deg = np.asarray(_U.sum(axis=1)).ravel()
P_T = (sp.diags(1 / np.maximum(_deg, 1)) @ _U).T.tocsr()   # random-walk transition matrix (transposed)

def personalized_pagerank(seed, alpha=0.5, iters=30):
    """Random walk that follows an edge with prob. alpha, otherwise jumps back to a node in proportion to `seed`.
    alpha=0.5 keeps the walk local (a few hops)."""
    s = seed / seed.sum()
    x = s.copy()
    for _ in range(iters):
        x = (1 - alpha) * s + alpha * (P_T @ x)
    return x

# --- Plain (global) PageRank: centrality, computed once on the directed graph ---
GLOBAL_PR = pd.Series(nx.pagerank(nx.from_pandas_edgelist(edges, "txId1", "txId2", create_using=nx.DiGraph)))
L["pagerank"] = GLOBAL_PR.reindex(L.index).values * len(GLOBAL_PR)    # rescaled so typical values are ~1
T["pagerank"] = GLOBAL_PR.reindex(T.index).values * len(GLOBAL_PR)

def ppr_from(p, alpha=0.5):
    """Fraud proximity: personalised PageRank seeded with predicted fraud probabilities p (indexed by txId).
    Unlabelled nodes get seed 0 but still pass the walk along."""
    idx = POS.reindex(p.index).values
    seed = np.zeros(len(ALL_IDS))
    seed[idx] = p.values
    x = personalized_pagerank(seed, alpha)
    own = (1 - alpha) * seed / seed.sum()                   # the part that is just the node's own seed
    return pd.DataFrame({"ppr": x[idx] * len(p),
                         "ppr_others": (x[idx] - own[idx]) * len(p)}, index=p.index)

def with_graph_ppr(fit, apply, alpha=0.5):
    """Submission 17's features + global PageRank + fraud-proximity PageRank (stage-1 computed once)."""
    p = stage1_preds(fit, apply)
    f = agg_nbr(p).join(ppr_from(p, alpha))
    base = feat_cols + STRUCT_COLS + ["pagerank"]
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

# Safety checks on validation (graph features have looked neutral here before, so look for "not clearly worse")
evaluate("S1 + global PageRank only, 3-seed", make_model=seed_avg,
         build=lambda f, a: (f[feat_cols + STRUCT_COLS + ["pagerank"]], a[feat_cols + STRUCT_COLS + ["pagerank"]]))
evaluate("S2 17's graph features + PageRank + fraud-proximity PPR, 3-seed",
         make_model=seed_avg, build=with_graph_ppr)

# Submission file + how different it is from 17
make_submission(lgbm5, with_graph_ppr, "21_lgbm5_graph_ppr")
p21 = pd.read_csv(f"{SUB_DIR}/21_lgbm5_graph_ppr.csv").set_index("index")["target"]
print(f"Rank correlation 21 vs 17: {spearmanr(p21, p17.loc[p21.index]).correlation:.4f}")

S1 + global PageRank only, 3-seed             folds=[0.9595 0.9055 0.997 ]  mean=0.9540
S2 17's graph features + PageRank + fraud-proximity PPR, 3-seed folds=[0.952  0.8764 0.9984]  mean=0.9423
21_lgbm5_graph_ppr           saved | rank correlation with 01 baseline: 0.8771
Rank correlation 21 vs 17: 0.9318


In [39]:
def with_graph_ppr_others(fit, apply, alpha=0.5):
    """17's features + fraud proximity from OTHER nodes only (own seed removed)."""
    p = stage1_preds(fit, apply)
    f = agg_nbr(p).join(ppr_from(p, alpha)[["ppr_others"]])
    base = feat_cols + STRUCT_COLS
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

evaluate("S3 17's features + PPR (others only, alpha 0.5)", make_model=seed_avg, build=with_graph_ppr_others)
evaluate("S4 17's features + PPR (others only, alpha 0.3)", make_model=seed_avg,
         build=lambda f, a: with_graph_ppr_others(f, a, alpha=0.3))

S3 17's features + PPR (others only, alpha 0.5) folds=[0.9532 0.8727 0.9985]  mean=0.9415
S4 17's features + PPR (others only, alpha 0.3) folds=[0.9537 0.8787 0.9985]  mean=0.9436


In [8]:
make_submission(lambda: seed_avg(seeds=tuple(range(10))), with_all_graph, "20_lgbm10_allgraph")

20_lgbm10_allgraph           saved | rank correlation with 01 baseline: 0.9518


In [9]:
# ===== RESTORE: everything needed for the graph-feature submissions (18 / 20 / 22) =====
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

SEED = 42
DATA_DIR, SUB_DIR = "../data", "../submissions"

# --- Data ---
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
train["label"] = pd.to_numeric(train["label"], errors="coerce")       # "none" -> NaN
feat_cols = [f"feat_{i}" for i in range(1, 166)]

lab = train[train["label"].notna()].copy()
lab["label"] = lab["label"].astype(int)

# --- Known nodes (labelled train + test) and structural features ---
nodes = pd.concat([lab, test.drop(columns="index")], ignore_index=True).set_index("txId").copy()
nodes["is_test"] = nodes.index.isin(test["txId"])
nodes["in_deg"]  = edges.groupby("txId2").size().reindex(nodes.index).fillna(0)
nodes["out_deg"] = edges.groupby("txId1").size().reindex(nodes.index).fillna(0)
nodes["deg"]     = nodes["in_deg"] + nodes["out_deg"]

e = edges[edges["txId1"].isin(nodes.index) & edges["txId2"].isin(nodes.index)]
nbr = pd.concat([
    pd.DataFrame({"node": e["txId2"].values, "nbr": e["txId1"].values, "dir": "in"}),
    pd.DataFrame({"node": e["txId1"].values, "nbr": e["txId2"].values, "dir": "out"}),
], ignore_index=True)
nodes["n_known_nbrs"]    = nbr.groupby("node").size().reindex(nodes.index).fillna(0)
nodes["frac_known_nbrs"] = nodes["n_known_nbrs"] / nodes["deg"].clip(lower=1)
STRUCT_COLS = ["in_deg", "out_deg", "deg", "n_known_nbrs", "frac_known_nbrs"]

L, T = nodes[~nodes["is_test"]].copy(), nodes[nodes["is_test"]].copy()

# --- Model + validation helpers ---
FOLDS = [(15, range(16, 26)), (20, range(21, 31)), (25, range(26, 36))]
LOG = []

def lgbm():
    return lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                              random_state=SEED, verbose=-1)

def evaluate(name, make_model=lgbm, build=lambda f, a: (f[feat_cols], a[feat_cols])):
    aucs = []
    for last_train, val_steps in FOLDS:
        fit = L[L["time_step"] <= last_train]
        va  = L[L["time_step"].isin(val_steps)]
        X_fit, X_va = build(fit, va)
        model = make_model().fit(X_fit, fit["label"])
        aucs.append(roc_auc_score(va["label"], model.predict_proba(X_va)[:, 1]))
    LOG.append({"experiment": name, **{f"fold{i+1}": round(a, 4) for i, a in enumerate(aucs)},
                "mean_AUC": round(np.mean(aucs), 4)})
    print(f"{name:45s} folds={np.round(aucs, 4)}  mean={np.mean(aucs):.4f}")

# --- Neighbour feature averages (C): top-20 features by importance on steps <= 15 ---
imp_model = lgbm().fit(L.loc[L["time_step"] <= 15, feat_cols], L.loc[L["time_step"] <= 15, "label"])
TOP = pd.Series(imp_model.feature_importances_, index=feat_cols).nlargest(20).index.tolist()
nb_vals = nodes.loc[nbr["nbr"], TOP].set_index(nbr["node"].values)
agg = nb_vals.groupby(level=0).agg(["mean", "max"])
agg.columns = [f"nbr_{stat}_{c}" for c, stat in agg.columns]
NBR_AGG_COLS = list(agg.columns)
nodes = nodes.drop(columns=NBR_AGG_COLS, errors="ignore").join(agg)
nodes[NBR_AGG_COLS] = nodes[NBR_AGG_COLS].fillna(-999)
L, T = nodes[~nodes["is_test"]].copy(), nodes[nodes["is_test"]].copy()

# --- Seed averaging ---
class AvgEnsemble:
    def __init__(self, makers): self.makers = makers
    def fit(self, X, y):
        self.models = [mk().fit(X, y) for mk in self.makers]
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

def seed_avg(params=None, seeds=(0, 1, 2)):
    params = params or {}
    return AvgEnsemble([lambda s=s: lgbm().set_params(random_state=s, **params) for s in seeds])

lgbm5 = lambda: seed_avg(seeds=(0, 1, 2, 3, 4))

# --- Neighbour-prediction features (1-hop and 2-hop), leakage-safe ---
def stage1_preds(fit, apply):
    p_fit = pd.Series(np.nan, index=fit.index)
    for tr_idx, oof_idx in GroupKFold(n_splits=5).split(fit, groups=fit["time_step"]):
        m = lgbm().fit(fit.iloc[tr_idx][feat_cols], fit.iloc[tr_idx]["label"])
        p_fit.iloc[oof_idx] = m.predict_proba(fit.iloc[oof_idx][feat_cols])[:, 1]
    m = lgbm().fit(fit[feat_cols], fit["label"])
    p_apply = pd.Series(m.predict_proba(apply[feat_cols])[:, 1], index=apply.index)
    return pd.concat([p_fit, p_apply])

def nbr_pred_features_v2(fit, apply):
    p = stage1_preds(fit, apply)
    nb = nbr[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    g = nb.groupby("node")["p"]
    out = pd.DataFrame({"nbr_p_max": g.max(), "nbr_p_mean": g.mean(), "nbr_p_sum": g.sum(),
                        "nbr_n_high": nb.assign(h=nb["p"] > 0.5).groupby("node")["h"].sum()})
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()
    pairs = nb[["node", "nbr"]]
    two = pairs.merge(pairs.rename(columns={"node": "nbr", "nbr": "nbr2"}), on="nbr")
    two = two[two["nbr2"] != two["node"]].drop_duplicates(["node", "nbr2"])
    two["p"] = p.reindex(two["nbr2"]).values
    g2 = two.groupby("node")["p"]
    out["nbr2_p_max"], out["nbr2_p_mean"], out["nbr2_n"] = g2.max(), g2.mean(), g2.size()
    return out.reindex(p.index).fillna(-1)

def with_all_graph(fit, apply):
    """Submission 18's features: raw + structure + neighbour averages + 1/2-hop neighbour predictions."""
    base = feat_cols + STRUCT_COLS + NBR_AGG_COLS
    f = nbr_pred_features_v2(fit, apply)
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

# --- Submission helper ---
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
base = pd.read_csv(f"{SUB_DIR}/01_baseline_LightGBM.csv").set_index("index")["target"]

def make_submission(make_model, build, tag):
    X_fit, X_test = build(L, T)
    model = make_model().fit(X_fit, L["label"])
    p = model.predict_proba(X_test.loc[test["txId"]])[:, 1]
    sub = pd.DataFrame({"index": test["index"].values, "target": p})
    assert len(sub) == len(sample) and set(sub["index"]) == set(sample["index"])
    assert sub["target"].between(0, 1).all()
    sub.to_csv(f"{SUB_DIR}/{tag}.csv", index=False)
    corr = spearmanr(sub.set_index("index")["target"], base.loc[sub["index"]]).correlation
    print(f"{tag:28s} saved | rank correlation with 01 baseline: {corr:.4f}")

print("Restored. L:", L.shape, "| T:", T.shape)

Restored. L: (31235, 213) | T: (15329, 213)


In [10]:
p18 = pd.read_csv(f"{SUB_DIR}/18_lgbm5_allgraph.csv").set_index("index")["target"]
p20 = pd.read_csv(f"{SUB_DIR}/20_lgbm10_allgraph.csv").set_index("index")["target"]
print(f"Rank correlation 20 vs 18: {spearmanr(p20, p18.loc[p20.index]).correlation:.4f}")

Rank correlation 20 vs 18: 0.9988


In [11]:
# ===== D5 + submission 22: neighbour feature averages over the TOP-40 features (instead of top-20) =====

# Feature ranking from the model trained on steps <= 15 only (same as experiment C), so no validation leakage
imp = pd.Series(imp_model.feature_importances_, index=feat_cols).sort_values(ascending=False)

def make_nbr_agg(k):
    """Mean and max of the top-k features over each node's known (labelled/test) neighbours.
    Uses feature values only, never labels, so it can be computed once for all nodes."""
    top = imp.head(k).index.tolist()
    vals = nodes.loc[nbr["nbr"], top].set_index(nbr["node"].values)
    agg = vals.groupby(level=0).agg(["mean", "max"])
    agg.columns = [f"nbr{k}_{stat}_{c}" for c, stat in agg.columns]
    return agg.reindex(nodes.index).fillna(-999)        # -999 = no known neighbour

agg40 = make_nbr_agg(40)
NBR40_COLS = list(agg40.columns)
L = L.drop(columns=NBR40_COLS, errors="ignore").join(agg40)
T = T.drop(columns=NBR40_COLS, errors="ignore").join(agg40)

def with_all_graph_40(fit, apply):
    """Like submission 18, but with neighbour averages over the top-40 features."""
    base = feat_cols + STRUCT_COLS + NBR40_COLS
    f = nbr_pred_features_v2(fit, apply)
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

# 1) Safety check on validation (submission 18's version, D3, scored 0.9490)
evaluate("D5 all graph, top-40 neighbour averages, 3-seed", make_model=seed_avg, build=with_all_graph_40)

# 2) Submission file
make_submission(lgbm5, with_all_graph_40, "22_lgbm5_allgraph40")

# 3) How different is it from 18?
p18 = pd.read_csv(f"{SUB_DIR}/18_lgbm5_allgraph.csv").set_index("index")["target"]
p22 = pd.read_csv(f"{SUB_DIR}/22_lgbm5_allgraph40.csv").set_index("index")["target"]
print(f"Rank correlation 22 vs 18: {spearmanr(p22, p18.loc[p22.index]).correlation:.4f}")

D5 all graph, top-40 neighbour averages, 3-seed folds=[0.9575 0.8927 0.9981]  mean=0.9494
22_lgbm5_allgraph40          saved | rank correlation with 01 baseline: 0.9466
Rank correlation 22 vs 18: 0.9947


In [12]:
import scipy.sparse as sp

# --- Full graph from the complete edge list (unlabelled nodes act as stepping stones) ---
ALL_IDS = pd.Index(pd.unique(edges[["txId1", "txId2"]].values.ravel()))
POS = pd.Series(np.arange(len(ALL_IDS)), index=ALL_IDS)
_r, _c = POS[edges["txId1"]].values, POS[edges["txId2"]].values
_A = sp.coo_matrix((np.ones(len(_r)), (_r, _c)), shape=(len(ALL_IDS),) * 2).tocsr()
_U = ((_A + _A.T) > 0).astype(float)
_deg = np.asarray(_U.sum(axis=1)).ravel()
P_T = (sp.diags(1 / np.maximum(_deg, 1)) @ _U).T.tocsr()

def personalized_pagerank(seed, alpha=0.3, iters=30):
    s = seed / seed.sum()
    x = s.copy()
    for _ in range(iters):
        x = (1 - alpha) * s + alpha * (P_T @ x)
    return x

def ppr_others(p, alpha=0.3):
    """Fraud proximity from OTHER nodes: PPR seeded with predicted fraud probabilities, own seed removed."""
    idx = POS.reindex(p.index).values
    seed = np.zeros(len(ALL_IDS))
    seed[idx] = p.values
    x = personalized_pagerank(seed, alpha)
    own = (1 - alpha) * seed / seed.sum()
    return pd.DataFrame({"ppr_others": (x[idx] - own[idx]) * len(p)}, index=p.index)

def agg_nbr(p):
    """Same 1-hop / 2-hop neighbour aggregates as nbr_pred_features_v2, from a given score p."""
    nb = nbr[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    g = nb.groupby("node")["p"]
    out = pd.DataFrame({"nbr_p_max": g.max(), "nbr_p_mean": g.mean(), "nbr_p_sum": g.sum(),
                        "nbr_n_high": nb.assign(h=nb["p"] > 0.5).groupby("node")["h"].sum()})
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()
    pairs = nb[["node", "nbr"]]
    two = pairs.merge(pairs.rename(columns={"node": "nbr", "nbr": "nbr2"}), on="nbr")
    two = two[two["nbr2"] != two["node"]].drop_duplicates(["node", "nbr2"])
    two["p"] = p.reindex(two["nbr2"]).values
    g2 = two.groupby("node")["p"]
    out["nbr2_p_max"], out["nbr2_p_mean"], out["nbr2_n"] = g2.max(), g2.mean(), g2.size()
    return out.reindex(p.index).fillna(-1)

def with_graph40_ppr(fit, apply):
    """Submission 22's features + fraud-proximity PageRank (stage-1 scores computed once)."""
    p = stage1_preds(fit, apply)
    f = agg_nbr(p).join(ppr_others(p))
    base = feat_cols + STRUCT_COLS + NBR40_COLS
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

make_submission(lgbm5, with_graph40_ppr, "23_lgbm5_graph40_ppr")
p23 = pd.read_csv(f"{SUB_DIR}/23_lgbm5_graph40_ppr.csv").set_index("index")["target"]
print(f"Rank correlation 23 vs 22: {spearmanr(p23, p22.loc[p23.index]).correlation:.4f}")

23_lgbm5_graph40_ppr         saved | rank correlation with 01 baseline: 0.8966
Rank correlation 23 vs 22: 0.9582


In [13]:
# ===== Pseudo-label UNLABELLED transactions (illicit only), on top of submission 22's graph features =====

# --- 1. Unlabelled training transactions and their static graph features (same definitions as for L/T) ---
U = train[train["label"].isna()].set_index("txId").copy()
U["in_deg"]  = edges.groupby("txId2").size().reindex(U.index).fillna(0)
U["out_deg"] = edges.groupby("txId1").size().reindex(U.index).fillna(0)
U["deg"]     = U["in_deg"] + U["out_deg"]

# Edges from each unlabelled node to its KNOWN (labelled/test) neighbours: the same neighbour set used for L/T
eu = edges[(edges["txId1"].isin(U.index) & edges["txId2"].isin(nodes.index)) |
           (edges["txId2"].isin(U.index) & edges["txId1"].isin(nodes.index))]
nbrU = pd.concat([pd.DataFrame({"node": eu["txId2"].values, "nbr": eu["txId1"].values, "dir": "in"}),
                  pd.DataFrame({"node": eu["txId1"].values, "nbr": eu["txId2"].values, "dir": "out"})],
                 ignore_index=True)
nbrU = nbrU[nbrU["node"].isin(U.index) & nbrU["nbr"].isin(nodes.index)]

U["n_known_nbrs"]    = nbrU.groupby("node").size().reindex(U.index).fillna(0)
U["frac_known_nbrs"] = U["n_known_nbrs"] / U["deg"].clip(lower=1)

top40 = imp.head(40).index.tolist()                                   # same top-40 features as NBR40_COLS
vals = nodes.loc[nbrU["nbr"], top40].set_index(nbrU["node"].values)
aggU = vals.groupby(level=0).agg(["mean", "max"])
aggU.columns = [f"nbr40_{stat}_{c}" for c, stat in aggU.columns]
U = U.join(aggU.reindex(U.index).fillna(-999)[NBR40_COLS])
print(f"Unlabelled rows: {len(U):,} | with ≥1 known neighbour: {(U['n_known_nbrs'] > 0).mean():.1%}")

# --- 2. Neighbour-prediction features for any set of nodes, from one stage-1 run ---
def agg_from_pairs(pairs, p, index):
    """1-hop and 2-hop neighbour aggregates of score p. `pairs` = (node, nbr, dir) edges from the target
    nodes to known nodes. Identical to nbr_pred_features_v2 for labelled/test nodes."""
    nb = pairs[pairs["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    g = nb.groupby("node")["p"]
    out = pd.DataFrame({"nbr_p_max": g.max(), "nbr_p_mean": g.mean(), "nbr_p_sum": g.sum(),
                        "nbr_n_high": nb.assign(h=nb["p"] > 0.5).groupby("node")["h"].sum()})
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()
    known = nbr.loc[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index), ["node", "nbr"]]
    two = nb[["node", "nbr"]].merge(known.rename(columns={"node": "nbr", "nbr": "nbr2"}), on="nbr")
    two = two[two["nbr2"] != two["node"]].drop_duplicates(["node", "nbr2"])
    two["p"] = p.reindex(two["nbr2"]).values
    g2 = two.groupby("node")["p"]
    out["nbr2_p_max"], out["nbr2_p_mean"], out["nbr2_n"] = g2.max(), g2.mean(), g2.size()
    return out.reindex(index).fillna(-1)

class WeightedSeeds:
    """LightGBM averaged over seeds, with sample weights."""
    def __init__(self, seeds): self.seeds = seeds
    def fit(self, X, y, w):
        self.models = [lgbm().set_params(random_state=s).fit(X, y, sample_weight=w) for s in self.seeds]
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

def pseudo_graph_predict(fit, apply, seeds=(0, 1, 2), thr=0.9, weight=0.5):
    """Train 22's model, pseudo-label confident ILLICIT unlabelled rows (steps <= latest training step),
    add them with lower weight, retrain, and predict `apply`. No pseudo-licit rows (those hurt before)."""
    p = stage1_preds(fit, apply)                                      # out-of-fold for fit rows (leakage-safe)
    base = feat_cols + STRUCT_COLS + NBR40_COLS
    Uf = U[U["time_step"] <= fit["time_step"].max()]
    Xf = fit[base].join(agg_from_pairs(nbr[nbr["node"].isin(fit.index)], p, fit.index))
    Xa = apply[base].join(agg_from_pairs(nbr[nbr["node"].isin(apply.index)], p, apply.index))
    Xu = Uf[base].join(agg_from_pairs(nbrU[nbrU["node"].isin(Uf.index)], p, Uf.index))

    y = fit["label"].values
    first = WeightedSeeds(seeds).fit(Xf, y, np.ones(len(y)))
    pu = first.predict_proba(Xu)[:, 1]
    pos = Xu[pu >= thr]                                               # confident pseudo-illicit only
    X2 = pd.concat([Xf, pos])
    y2 = np.r_[y, np.ones(len(pos))]
    w2 = np.r_[np.ones(len(y)), np.full(len(pos), weight)]
    final = WeightedSeeds(seeds).fit(X2, y2, w2)
    print(f"   steps <= {fit['time_step'].max()}: +{len(pos)} pseudo-illicit (real illicit {int(y.sum())})")
    return pd.Series(final.predict_proba(Xa)[:, 1], index=apply.index)

def evaluate_pseudo(name, **kw):
    aucs = []
    for last_train, val_steps in FOLDS:
        fit, va = L[L["time_step"] <= last_train], L[L["time_step"].isin(val_steps)]
        pr = pseudo_graph_predict(fit, va, **kw)
        aucs.append(roc_auc_score(va["label"], pr.loc[va.index]))
    print(f"{name:45s} folds={np.round(aucs, 4)}  mean={np.mean(aucs):.4f}")

# --- 3. Validation: compare with 22's pipeline (D5 = 0.9494) ---
evaluate_pseudo("P1 22-graph + pseudo-illicit (p>=0.9, w=0.5)")
evaluate_pseudo("P2 22-graph + pseudo-illicit (p>=0.95, w=0.5)", thr=0.95)

# --- 4. Submission 24 (5 seeds, all labelled rows, all unlabelled rows up to step 35) ---
pr = pseudo_graph_predict(L, T, seeds=(0, 1, 2, 3, 4), thr=0.9, weight=0.5)
sub = pd.DataFrame({"index": test["index"].values, "target": pr.loc[test["txId"]].values})
assert len(sub) == len(sample) and set(sub["index"]) == set(sample["index"]) and sub["target"].between(0, 1).all()
sub.to_csv(f"{SUB_DIR}/24_lgbm5_graph40_pseudoillicit.csv", index=False)
p22 = pd.read_csv(f"{SUB_DIR}/22_lgbm5_allgraph40.csv").set_index("index")["target"]
print(f"Saved 24 | rank correlation with 22: {spearmanr(sub.set_index('index')['target'], p22.loc[sub['index']]).correlation:.4f}")

Unlabelled rows: 110,537 | with ≥1 known neighbour: 29.4%
   steps <= 15: +1230 pseudo-illicit (real illicit 1152)
   steps <= 20: +2097 pseudo-illicit (real illicit 1771)
   steps <= 25: +3039 pseudo-illicit (real illicit 2337)
P1 22-graph + pseudo-illicit (p>=0.9, w=0.5)  folds=[0.9588 0.891  0.9978]  mean=0.9492
   steps <= 15: +1077 pseudo-illicit (real illicit 1152)
   steps <= 20: +1686 pseudo-illicit (real illicit 1771)
   steps <= 25: +2481 pseudo-illicit (real illicit 2337)
P2 22-graph + pseudo-illicit (p>=0.95, w=0.5) folds=[0.9574 0.8976 0.9979]  mean=0.9510
   steps <= 35: +3653 pseudo-illicit (real illicit 3644)
Saved 24 | rank correlation with 22: 0.9935


In [14]:
import scipy.sparse as sp

# --- Fraud-proximity PageRank over the FULL graph (unlabelled nodes included) ---
ALL_IDS = pd.Index(pd.unique(edges[["txId1", "txId2"]].values.ravel()))
POS = pd.Series(np.arange(len(ALL_IDS)), index=ALL_IDS)
_r, _c = POS[edges["txId1"]].values, POS[edges["txId2"]].values
_A = sp.coo_matrix((np.ones(len(_r)), (_r, _c)), shape=(len(ALL_IDS),) * 2).tocsr()
_U = ((_A + _A.T) > 0).astype(float)
P_T = (sp.diags(1 / np.maximum(np.asarray(_U.sum(axis=1)).ravel(), 1)) @ _U).T.tocsr()

def personalized_pagerank(seed, alpha=0.3, iters=30):
    s = seed / seed.sum()
    x = s.copy()
    for _ in range(iters):
        x = (1 - alpha) * s + alpha * (P_T @ x)
    return x

def pseudo_graph_predict_v2(fit, apply, seeds=(0, 1, 2), thr=0.9, weight=0.5, filt="nbr", ppr_q=0.9):
    """Like pseudo_graph_predict, but a confident unknown is only added if the GRAPH also points to fraud:
       filt="nbr": it has >= 1 known neighbour with stage-1 fraud score >= 0.5
       filt="ppr": its fraud-proximity PageRank is in the top (1 - ppr_q) of unknowns"""
    p = stage1_preds(fit, apply)
    base = feat_cols + STRUCT_COLS + NBR40_COLS
    Uf = U[U["time_step"] <= fit["time_step"].max()]
    Xf = fit[base].join(agg_from_pairs(nbr[nbr["node"].isin(fit.index)], p, fit.index))
    Xa = apply[base].join(agg_from_pairs(nbr[nbr["node"].isin(apply.index)], p, apply.index))
    Xu = Uf[base].join(agg_from_pairs(nbrU[nbrU["node"].isin(Uf.index)], p, Uf.index))

    y = fit["label"].values
    pu = WeightedSeeds(seeds).fit(Xf, y, np.ones(len(y))).predict_proba(Xu)[:, 1]
    keep = pu >= thr
    if filt == "nbr":
        keep &= Xu["nbr_p_max"].values >= 0.5                        # -1 (no known neighbour) is excluded
    elif filt == "ppr":
        seed = np.zeros(len(ALL_IDS))
        seed[POS.reindex(p.index).values] = p.values
        ppr_u = personalized_pagerank(seed)[POS.reindex(Uf.index).values]
        keep &= ppr_u >= np.quantile(ppr_u, ppr_q)

    pos = Xu[keep]
    X2 = pd.concat([Xf, pos])
    y2 = np.r_[y, np.ones(len(pos))]
    w2 = np.r_[np.ones(len(y)), np.full(len(pos), weight)]
    print(f"   steps <= {fit['time_step'].max()}: +{len(pos)} pseudo-illicit (filter={filt}; "
          f"{(pos['n_known_nbrs'] > 0).mean():.0%} have a known neighbour)")
    return pd.Series(WeightedSeeds(seeds).fit(X2, y2, w2).predict_proba(Xa)[:, 1], index=apply.index)

def evaluate_pseudo2(name, **kw):
    aucs = []
    for last_train, val_steps in FOLDS:
        fit, va = L[L["time_step"] <= last_train], L[L["time_step"].isin(val_steps)]
        aucs.append(roc_auc_score(va["label"], pseudo_graph_predict_v2(fit, va, **kw).loc[va.index]))
    print(f"{name:45s} folds={np.round(aucs, 4)}  mean={np.mean(aucs):.4f}")

# Validation (22's pipeline D5 = 0.9494; unfiltered pseudo-illicit P1 = 0.9492)
evaluate_pseudo2("P3 pseudo-illicit + suspicious known neighbour", filt="nbr")
evaluate_pseudo2("P4 pseudo-illicit + top-10% fraud-proximity PPR", filt="ppr")

   steps <= 15: +136 pseudo-illicit (filter=nbr; 100% have a known neighbour)
   steps <= 20: +230 pseudo-illicit (filter=nbr; 100% have a known neighbour)
   steps <= 25: +320 pseudo-illicit (filter=nbr; 100% have a known neighbour)
P3 pseudo-illicit + suspicious known neighbour folds=[0.9574 0.8857 0.9981]  mean=0.9471
   steps <= 15: +663 pseudo-illicit (filter=ppr; 44% have a known neighbour)
   steps <= 20: +1014 pseudo-illicit (filter=ppr; 49% have a known neighbour)
   steps <= 25: +1224 pseudo-illicit (filter=ppr; 53% have a known neighbour)
P4 pseudo-illicit + top-10% fraud-proximity PPR folds=[0.9589 0.8949 0.998 ]  mean=0.9506


In [15]:
pr = pseudo_graph_predict_v2(L, T, seeds=(0, 1, 2, 3, 4), filt="ppr")
sub = pd.DataFrame({"index": test["index"].values, "target": pr.loc[test["txId"]].values})
assert len(sub) == len(sample) and set(sub["index"]) == set(sample["index"]) and sub["target"].between(0, 1).all()
sub.to_csv(f"{SUB_DIR}/25_lgbm5_graph40_pseudo_ppr.csv", index=False)
print(f"Rank correlation 25 vs 22: {spearmanr(sub.set_index('index')['target'], p22.loc[sub['index']]).correlation:.4f}")

   steps <= 35: +1408 pseudo-illicit (filter=ppr; 58% have a known neighbour)
Rank correlation 25 vs 22: 0.9950


In [16]:
%pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

class SMOTEModel:
    """Oversamples fraud with SMOTE on the TRAINING rows of each fold only (never validation/test)."""
    def __init__(self, maker, ratio=0.3):
        self.maker, self.ratio = maker, ratio      # ratio = fraud / licit after resampling (currently ~0.13)
    def fit(self, X, y):
        Xr, yr = SMOTE(sampling_strategy=self.ratio, k_neighbors=5, random_state=SEED).fit_resample(X, y)
        self.m = self.maker().fit(Xr, yr)
        return self
    def predict_proba(self, X):
        return self.m.predict_proba(X)

for r in [0.3, 0.5, 1.0]:
    evaluate(f"T SMOTE fraud/licit={r}, LGBM 3-seed (raw)", make_model=lambda r=r: SMOTEModel(seed_avg, r))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
T SMOTE fraud/licit=0.3, LGBM 3-seed (raw)    folds=[0.9603 0.9015 0.9972]  mean=0.9530
T SMOTE fraud/licit=0.5, LGBM 3-seed (raw)    folds=[0.9563 0.913  0.9971]  mean=0.9555
T SMOTE fraud/licit=1.0, LGBM 3-seed (raw)    folds=[0.9597 0.9133 0.9971]  mean=0.9567


In [17]:
# Validation: 22's pipeline (D5 = 0.9494) with SMOTE on the training rows of each fold
evaluate("T4 22-graph + SMOTE 0.5, 3-seed", make_model=lambda: SMOTEModel(seed_avg, 0.5), build=with_all_graph_40)
evaluate("T5 22-graph + SMOTE 1.0, 3-seed", make_model=lambda: SMOTEModel(seed_avg, 1.0), build=with_all_graph_40)


T4 22-graph + SMOTE 0.5, 3-seed               folds=[0.9626 0.9009 0.9982]  mean=0.9539
T5 22-graph + SMOTE 1.0, 3-seed               folds=[0.9663 0.9072 0.9982]  mean=0.9572


In [18]:
make_submission(lambda: SMOTEModel(lgbm5, 1.0), with_all_graph_40, "26_lgbm5_graph40_smote")   # or 0.5
p26 = pd.read_csv(f"{SUB_DIR}/26_lgbm5_graph40_smote.csv").set_index("index")["target"]
p22 = pd.read_csv(f"{SUB_DIR}/22_lgbm5_allgraph40.csv").set_index("index")["target"]
print(f"Rank correlation 26 vs 22: {spearmanr(p26, p22.loc[p26.index]).correlation:.4f}")

26_lgbm5_graph40_smote       saved | rank correlation with 01 baseline: 0.9195
Rank correlation 26 vs 22: 0.9802


In [19]:
class SMOTEBag:
    """Each seed gets its own SMOTE sample + its own LightGBM; predictions are averaged."""
    def __init__(self, seeds=(0, 1, 2, 3, 4), ratio=1.0):
        self.seeds, self.ratio = seeds, ratio
    def fit(self, X, y):
        self.models = []
        for s in self.seeds:
            Xr, yr = SMOTE(sampling_strategy=self.ratio, k_neighbors=5, random_state=s).fit_resample(X, y)
            self.models.append(lgbm().set_params(random_state=s).fit(Xr, yr))
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

evaluate("T6 22-graph + SMOTE-bag 1.0, 3 seeds", make_model=lambda: SMOTEBag(seeds=(0, 1, 2)), build=with_all_graph_40)

make_submission(lambda: SMOTEBag(), with_all_graph_40, "27_lgbm5_graph40_smotebag")
p27 = pd.read_csv(f"{SUB_DIR}/27_lgbm5_graph40_smotebag.csv").set_index("index")["target"]
print(f"Rank correlation 27 vs 26: {spearmanr(p27, p26.loc[p27.index]).correlation:.4f}")


T6 22-graph + SMOTE-bag 1.0, 3 seeds          folds=[0.9638 0.8985 0.9982]  mean=0.9535
27_lgbm5_graph40_smotebag    saved | rank correlation with 01 baseline: 0.9238
Rank correlation 27 vs 26: 0.9957
